In [12]:
table_names = [
    "patients",
    "encounters",
    "claims",
    "claims_transactions",
    "payers"
]

for table_name in table_names:
    print(f"\n--- silver.{table_name} ---")
    spark.table(f"silver.{table_name}").printSchema()


StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 14, Finished, Available, Finished, False)


--- silver.patients ---
root
 |-- id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- deathdate: date (nullable = true)
 |-- ssn: string (nullable = true)
 |-- drivers: string (nullable = true)
 |-- passport: string (nullable = true)
 |-- prefix: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- suffix: string (nullable = true)
 |-- maiden: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- healthcare_expenses: decimal(18,2) (nullable = true)
 |-- healthcare_coverage: decimal(18,2) (nullabl

In [13]:
from pyspark.sql import functions as F

patients_df = spark.table("silver.patients")
encounters_df = spark.table("silver.encounters")
claims_df = spark.table("silver.claims")
transactions_df = spark.table("silver.claims_transactions")
payers_df = spark.table("silver.payers")

table_checks = [
    ("patients", patients_df, "id"),
    ("encounters", encounters_df, "encounter_id"),
    ("claims", claims_df, "claim_id"),
    ("claims_transactions", transactions_df, "transaction_id"),
    ("payers", payers_df, "payer_id")
]

results = []

for table_name, df, key in table_checks:
    summary = df.agg(
        F.count("*").alias("row_count"),
        F.count(key).alias("non_null_keys"),
        F.countDistinct(key).alias("distinct_keys")
    ).first()

    results.append((
        table_name,
        summary["row_count"],
        summary["row_count"] - summary["non_null_keys"],
        summary["non_null_keys"] - summary["distinct_keys"]
    ))

key_audit_df = spark.createDataFrame(
    results,
    """
    table_name STRING,
    row_count LONG,
    null_key_count LONG,
    duplicate_key_excess LONG
    """
)

display(key_audit_df.orderBy("table_name"))

assert all(
    null_count == 0 and duplicate_count == 0
    for _, _, null_count, duplicate_count in results
), "Primary-key validation failed. Review the audit output."

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 215778e3-3840-4999-9394-44dee2911a27)

In [14]:
# Validate required relationships and optional payer relationships
relationship_checks = [
    (
        "Claims → patients",
        claims_df, "patient_id",
        patients_df, "id",
        True
    ),
    (
        "Claims → encounters",
        claims_df, "encounter_id",
        encounters_df, "encounter_id",
        True
    ),
    (
        "Transactions → claims",
        transactions_df, "claim_id",
        claims_df, "claim_id",
        True
    ),
    (
        "Transactions → patients",
        transactions_df, "patient_id",
        patients_df, "id",
        True
    ),
    (
        "Transactions → encounters",
        transactions_df, "encounter_id",
        encounters_df, "encounter_id",
        True
    ),
    (
        "Encounters → patients",
        encounters_df, "patient_id",
        patients_df, "id",
        True
    ),
    (
        "Claims → primary payers",
        claims_df, "primary_payer_id",
        payers_df, "payer_id",
        False
    ),
    (
        "Claims → secondary payers",
        claims_df, "secondary_payer_id",
        payers_df, "payer_id",
        False
    )
]

relationship_results = []

for name, child_df, child_key, parent_df, parent_key, required in relationship_checks:

    missing_count = child_df.filter(
        F.col(child_key).isNull()
    ).count()

    parent_keys = parent_df.select(
        F.col(parent_key).alias("_valid_key")
    ).distinct()

    unmatched_count = (
        child_df
        .filter(F.col(child_key).isNotNull())
        .join(
            parent_keys,
            F.col(child_key) == F.col("_valid_key"),
            "left_anti"
        )
        .count()
    )

    relationship_results.append((
        name,
        missing_count,
        unmatched_count,
        required
    ))

relationship_audit_df = spark.createDataFrame(
    relationship_results,
    """
    relationship STRING,
    null_reference_count LONG,
    unmatched_reference_count LONG,
    reference_required BOOLEAN
    """
)

display(relationship_audit_df)

assert all(
    unmatched == 0 and (not required or missing == 0)
    for _, missing, unmatched, required in relationship_results
), "Relationship validation failed. Review the audit output."

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 633b5b0c-6f5e-4e4f-8350-bf3d78676bad)

In [15]:
# 1. Compare each claim's patient with its encounter's patient
claim_encounter_check_df = (
    claims_df.alias("c")
    .join(
        encounters_df.alias("e"),
        F.col("c.encounter_id") == F.col("e.encounter_id"),
        "inner"
    )
)

claim_encounter_mismatches = (
    claim_encounter_check_df
    .filter(
        ~F.col("c.patient_id").eqNullSafe(F.col("e.patient_id"))
    )
    .count()
)

# 2. Compare each transaction with its claim
transaction_claim_check_df = (
    transactions_df.alias("t")
    .join(
        claims_df.alias("c"),
        F.col("t.claim_id") == F.col("c.claim_id"),
        "inner"
    )
)

transaction_mismatch_summary = (
    transaction_claim_check_df
    .agg(
        F.count(
            F.when(
                ~F.col("t.patient_id").eqNullSafe(
                    F.col("c.patient_id")
                ),
                1
            )
        ).alias("patient_mismatches"),

        F.count(
            F.when(
                ~F.col("t.encounter_id").eqNullSafe(
                    F.col("c.encounter_id")
                ),
                1
            )
        ).alias("encounter_mismatches")
    )
    .first()
)

consistency_results = [
    (
        "Claim patient matches encounter patient",
        claim_encounter_mismatches
    ),
    (
        "Transaction patient matches claim patient",
        transaction_mismatch_summary["patient_mismatches"]
    ),
    (
        "Transaction encounter matches claim encounter",
        transaction_mismatch_summary["encounter_mismatches"]
    )
]

consistency_audit_df = spark.createDataFrame(
    consistency_results,
    "validation_check STRING, mismatch_count LONG"
)

display(consistency_audit_df)

assert all(
    mismatch_count == 0
    for _, mismatch_count in consistency_results
), "Linked records disagree. Review the consistency audit."

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 367e3e19-670f-43e5-b8ce-be1339113f9b)

In [16]:
is_charge = F.col("transaction_type") == "CHARGE"
is_payment = F.col("transaction_type") == "PAYMENT"

zero_amount = F.lit(0).cast("decimal(18,2)")

claim_transaction_summary_df = (
    transactions_df
    .groupBy("claim_id")
    .agg(
        F.count("*").alias("transaction_count"),

        F.sum(
            F.when(is_charge, 1).otherwise(0)
        ).alias("charge_count"),

        F.sum(
            F.when(is_payment, 1).otherwise(0)
        ).alias("payment_count"),

        F.sum(
            F.when(
                F.col("transaction_type").isin(
                    "TRANSFERIN", "TRANSFEROUT"
                ),
                1
            ).otherwise(0)
        ).alias("transfer_count"),

        F.sum(
            F.when(
                is_charge,
                F.coalesce(
                    F.col("transaction_amount"),
                    zero_amount
                )
            ).otherwise(zero_amount)
        ).alias("total_charge_amount"),

        F.sum(
            F.when(
                is_payment,
                F.coalesce(
                    F.col("payment_amount"),
                    zero_amount
                )
            ).otherwise(zero_amount)
        ).alias("total_payment_amount"),

        F.countDistinct(
            F.when(is_charge, F.col("procedure_code"))
        ).alias("unique_charge_procedure_count"),

        F.sum(
            F.when(
                is_charge & F.col("transaction_amount").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_charge_amount_count"),

        F.sum(
            F.when(
                is_payment & F.col("payment_amount").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_payment_amount_count")
    )
)

display(
    claim_transaction_summary_df
    .orderBy("claim_id")
    .limit(10)
)

# Confirm every claim has a transaction summary
claims_without_transactions = (
    claims_df.select("claim_id")
    .join(
        claim_transaction_summary_df.select("claim_id"),
        "claim_id",
        "left_anti"
    )
    .count()
)

summary_claim_count = claim_transaction_summary_df.count()
source_claim_count = claims_df.count()

summarized_transaction_count = (
    claim_transaction_summary_df
    .agg(F.sum("transaction_count").alias("total"))
    .first()["total"]
)

source_transaction_count = transactions_df.count()

print(f"Source claims: {source_claim_count:,}")
print(f"Summarized claims: {summary_claim_count:,}")
print(f"Claims without transactions: {claims_without_transactions:,}")
print(f"Source transactions: {source_transaction_count:,}")
print(f"Summarized transactions: {summarized_transaction_count:,}")

assert claims_without_transactions == 0
assert summary_claim_count == source_claim_count
assert summarized_transaction_count == source_transaction_count

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43dc6fee-3297-47dc-a658-d6f5c3ec82d0)

Source claims: 117,889
Summarized claims: 117,889
Claims without transactions: 0
Source transactions: 711,238
Summarized transactions: 711,238


In [17]:
# Count populated diagnosis fields for each claim
diagnosis_count_expression = sum(
    F.when(
        F.col(f"diagnosis_code_{i}").isNotNull(),
        1
    ).otherwise(0)
    for i in range(1, 9)
)

claim_header_df = (
    claims_df
    .select(
        "claim_id",
        "patient_id",
        "encounter_id",
        "provider_id",
        "primary_payer_id",
        "secondary_payer_id",
        "service_date",
        "claim_coverage_category",
        "primary_claim_status",
        "secondary_claim_status",
        "patient_claim_status",
        "total_outstanding_amount",
        *[f"diagnosis_code_{i}" for i in range(1, 9)]
    )
    .withColumn(
        "diagnosis_count",
        diagnosis_count_expression
    )
)

# Both DataFrames should have exactly one row per claim
claim_features_df = (
    claim_header_df
    .join(
        claim_transaction_summary_df,
        on="claim_id",
        how="left"
    )
)

# Validate that the join preserved one row per claim
join_summary = (
    claim_features_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("claim_id").alias("distinct_claim_count"),
        F.count(
            F.when(F.col("transaction_count").isNull(), 1)
        ).alias("missing_transaction_summary_count")
    )
    .first()
)

print(f"Joined rows: {join_summary['row_count']:,}")
print(
    f"Distinct claims: "
    f"{join_summary['distinct_claim_count']:,}"
)
print(
    f"Missing transaction summaries: "
    f"{join_summary['missing_transaction_summary_count']:,}"
)

assert join_summary["row_count"] == source_claim_count
assert join_summary["distinct_claim_count"] == source_claim_count
assert join_summary["missing_transaction_summary_count"] == 0

display(
    claim_features_df
    .select(
        "claim_id",
        "service_date",
        "diagnosis_count",
        "transaction_count",
        "charge_count",
        "payment_count",
        "total_charge_amount",
        "total_payment_amount",
        "total_outstanding_amount"
    )
    .orderBy("claim_id")
    .limit(10)
)

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 19, Finished, Available, Finished, False)

Joined rows: 117,889
Distinct claims: 117,889
Missing transaction summaries: 0


SynapseWidget(Synapse.DataFrame, 229fb32e-f7d0-49b9-be35-827d609bf286)

In [18]:
# Select only the encounter information needed
encounter_context_df = encounters_df.select(
    "encounter_id",
    "encounter_class",
    "encounter_code",
    "encounter_duration_minutes"
)

# Rename patients.id to match the claim's patient_id
patient_context_df = patients_df.select(
    F.col("id").alias("patient_id"),
    "birthdate",
    "deathdate"
)

claim_features_enriched_df = (
    claim_features_df
    .join(
        encounter_context_df,
        on="encounter_id",
        how="left"
    )
    .join(
        patient_context_df,
        on="patient_id",
        how="left"
    )
    .withColumn(
        "age_at_service_years",
        F.round(
            F.datediff("service_date", "birthdate") / F.lit(365.2425),
            2
        )
    )
    .withColumn(
        "service_before_birth_flag",
        F.when(
            F.col("service_date").isNull()
            | F.col("birthdate").isNull(),
            F.lit(None).cast("integer")
        )
        .when(F.col("service_date") < F.col("birthdate"), 1)
        .otherwise(0)
    )
    .withColumn(
        "service_after_death_flag",
        F.when(
            F.col("service_date").isNull(),
            F.lit(None).cast("integer")
        )
        .when(
            F.col("deathdate").isNotNull()
            & (F.col("service_date") > F.col("deathdate")),
            1
        )
        .otherwise(0)
    )
)

# Confirm the joins preserved one row per claim
enrichment_summary = (
    claim_features_enriched_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("claim_id").alias("distinct_claim_count"),
        F.count(
            F.when(F.col("birthdate").isNull(), 1)
        ).alias("missing_birthdate_count"),
        F.count(
            F.when(F.col("encounter_class").isNull(), 1)
        ).alias("missing_encounter_class_count")
    )
    .first()
)

print(enrichment_summary)

assert enrichment_summary["row_count"] == source_claim_count
assert enrichment_summary["distinct_claim_count"] == source_claim_count

display(
    claim_features_enriched_df
    .select(
        "claim_id",
        "service_date",
        "encounter_class",
        "encounter_duration_minutes",
        "age_at_service_years",
        "total_charge_amount",
        "total_payment_amount",
        "service_before_birth_flag",
        "service_after_death_flag"
    )
    .orderBy("claim_id")
    .limit(10)
)

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 20, Finished, Available, Finished, False)

Row(row_count=117889, distinct_claim_count=117889, missing_birthdate_count=0, missing_encounter_class_count=0)


SynapseWidget(Synapse.DataFrame, 92cf9b7f-758e-4b8f-bef6-8e15d03029b3)

In [19]:
# Add financial comparison features
claim_features_final_df = (
    claim_features_enriched_df
    .withColumn(
        "payment_minus_charge_amount",
        F.col("total_payment_amount")
        - F.col("total_charge_amount")
    )
    .withColumn(
        "payment_to_charge_ratio",
        F.when(
            F.col("total_charge_amount") > 0,
            F.col("total_payment_amount")
            / F.col("total_charge_amount")
        )
    )
    .withColumn(
        "payment_exceeds_charge_flag",
        F.when(
            F.col("total_payment_amount").isNull()
            | F.col("total_charge_amount").isNull(),
            F.lit(None).cast("integer")
        )
        .when(
            F.col("payment_minus_charge_amount")
            > F.lit("0.01").cast("decimal(18,2)"),
            1
        )
        .otherwise(0)
    )
)

# Independently calculate totals directly from Silver transactions
source_financial_totals = (
    transactions_df
    .agg(
        F.sum(
            F.when(
                F.col("transaction_type") == "CHARGE",
                F.coalesce(
                    F.col("transaction_amount"),
                    zero_amount
                )
            ).otherwise(zero_amount)
        ).alias("source_charge_total"),

        F.sum(
            F.when(
                F.col("transaction_type") == "PAYMENT",
                F.coalesce(
                    F.col("payment_amount"),
                    zero_amount
                )
            ).otherwise(zero_amount)
        ).alias("source_payment_total")
    )
    .first()
)

feature_totals = (
    claim_features_final_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("claim_id").alias("distinct_claim_count"),
        F.sum("total_charge_amount").alias("feature_charge_total"),
        F.sum("total_payment_amount").alias("feature_payment_total"),
        F.sum("payment_exceeds_charge_flag").alias(
            "payment_exceeds_charge_count"
        ),
        F.sum("service_before_birth_flag").alias(
            "service_before_birth_count"
        ),
        F.sum("service_after_death_flag").alias(
            "service_after_death_count"
        )
    )
    .first()
)

assert feature_totals["row_count"] == source_claim_count
assert feature_totals["distinct_claim_count"] == source_claim_count

assert (
    feature_totals["feature_charge_total"]
    == source_financial_totals["source_charge_total"]
), "Charge totals changed during aggregation or joins."

assert (
    feature_totals["feature_payment_total"]
    == source_financial_totals["source_payment_total"]
), "Payment totals changed during aggregation or joins."

print(f"Claims: {feature_totals['row_count']:,}")
print(f"Total charges: ${feature_totals['feature_charge_total']:,.2f}")
print(f"Total payments: ${feature_totals['feature_payment_total']:,.2f}")
print(
    "Payments exceed charges: "
    f"{feature_totals['payment_exceeds_charge_count']:,}"
)
print(
    "Service before birth: "
    f"{feature_totals['service_before_birth_count']:,}"
)
print(
    "Service after death: "
    f"{feature_totals['service_after_death_count']:,}"
)

print("Row counts and financial reconciliation passed.")

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 21, Finished, Available, Finished, False)

Claims: 117,889
Total charges: $206,901,374.78
Total payments: $207,482,352.15
Payments exceed charges: 13,114
Service before birth: 0
Service after death: 180
Row counts and financial reconciliation passed.


In [20]:
example_claim_id = "00009d0c-a000-5263-7272-88aad3f53a09"

# Display the claim-level summary
display(
    claim_features_final_df
    .filter(F.col("claim_id") == example_claim_id)
    .select(
        "claim_id",
        "service_date",
        "encounter_class",
        "charge_count",
        "payment_count",
        "total_charge_amount",
        "total_payment_amount",
        "payment_minus_charge_amount",
        "total_outstanding_amount",
        "payment_exceeds_charge_flag"
    )
)

# Display the financial entries behind that summary
display(
    transactions_df
    .filter(F.col("claim_id") == example_claim_id)
    .select(
        "transaction_id",
        "charge_id",
        "transaction_type",
        "procedure_code",
        "units",
        "transaction_amount",
        "payment_amount",
        "adjustment_amount",
        "transfer_amount",
        "outstanding_amount",
        "payment_method",
        "service_start_timestamp"
    )
    .orderBy("transaction_type", "transaction_id")
)

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4b528d2f-707f-4c5a-bb60-6f46a3f22e49)

SynapseWidget(Synapse.DataFrame, 37124cb5-ed9b-4d8b-b000-9a65b12dd882)

In [21]:
target_table = "gold.claim_anomaly_features"

# Creates or replaces this project's feature table
(
    claim_features_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# Read back and verify the saved table
saved_features_df = spark.table(target_table)

saved_summary = saved_features_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("claim_id").alias("distinct_claim_count"),
    F.sum("total_charge_amount").alias("total_charges"),
    F.sum("total_payment_amount").alias("total_payments")
).first()

assert (
    saved_summary["row_count"] == source_claim_count
), "Saved row count does not match source claims."

assert (
    saved_summary["distinct_claim_count"] == source_claim_count
), "Saved claim IDs are not unique or claims are missing."

assert (
    saved_summary["total_charges"]
    == source_financial_totals["source_charge_total"]
), "Saved charge total does not match the source."

assert (
    saved_summary["total_payments"]
    == source_financial_totals["source_payment_total"]
), "Saved payment total does not match the source."

print(f"Saved table: {target_table}")
print(f"Validated claims: {saved_summary['row_count']:,}")
print("Saved financial totals match the source.")

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 23, Finished, Available, Finished, False)

Saved table: gold.claim_anomaly_features
Validated claims: 117,889
Saved financial totals match the source.


In [22]:
export_folder = "Files/cotiviti/claim_anomaly_features_export"

(
    spark.table("gold.claim_anomaly_features")
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(export_folder)
)

print(f"Export completed: {export_folder}")

StatementMeta(, 193f56db-a2a3-4ced-9863-bb84238b0b56, 24, Finished, Available, Finished, False)

Export completed: Files/cotiviti/claim_anomaly_features_export
